# 🎤 RVC 声質クローン学習 - Google Colab

このノートブックで stand.fm の音声からRVCモデルを学習します。

**前提条件:**
- ランタイムを **GPU** に設定してください (ランタイム → ランタイムのタイプを変更 → T4 GPU)
- Google Drive をマウントしてデータセットをアップロードしてください


In [ ]:
# GPU確認
!nvidia-smi

In [ ]:
# Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

## Step 1: RVC WebUI インストール

In [ ]:
import os
os.chdir('/content')

!git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git rvc
os.chdir('/content/rvc')

!pip install -r requirements.txt -q
!pip install tensorboard -q

# 必要なモデルをダウンロード
!apt-get install -y aria2 -q
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
    https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/D40k.pth \
    -d /content/rvc/pretrained_v2 -o D40k.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
    https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/G40k.pth \
    -d /content/rvc/pretrained_v2 -o G40k.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
    https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth \
    -d /content/rvc/pretrained_v2 -o f0D40k.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
    https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth \
    -d /content/rvc/pretrained_v2 -o f0G40k.pth

# hubert モデル
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
    https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt \
    -d /content/rvc -o hubert_base.pt

# rmvpe モデル（高精度ピッチ抽出）
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
    https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt \
    -d /content/rvc -o rmvpe.pt

print('✓ RVC インストール完了')

## Step 2: データセットのアップロード

`2_preprocess_audio.py` で生成した `audio/dataset/` フォルダの内容を
Google Drive の `/MyDrive/rvc_dataset/` にアップロードしてください。

In [ ]:
import shutil
from pathlib import Path

# モデル名（任意）
MODEL_NAME = "standfm_voice"  # ← 変更可

# Drive からデータセットをコピー
drive_dataset = Path(f"/content/drive/MyDrive/rvc_dataset")
local_dataset = Path(f"/content/rvc/dataset/{MODEL_NAME}")
local_dataset.mkdir(parents=True, exist_ok=True)

if drive_dataset.exists():
    wav_files = list(drive_dataset.glob("*.wav"))
    for f in wav_files:
        shutil.copy(f, local_dataset / f.name)
    print(f"✓ {len(wav_files)} ファイルをコピー")
else:
    print(f"⚠ Drive にデータセットが見つかりません: {drive_dataset}")
    print("手動でファイルをアップロードしてください")

# ファイル数確認
files = list(local_dataset.glob("*.wav"))
print(f"データセット: {len(files)} ファイル")
total_sec = len(files) * 8  # 8秒/セグメント（概算）
print(f"推定総時間: {total_sec // 60} 分 {total_sec % 60} 秒")

## Step 3: 音声特徴量の抽出

In [ ]:
import os
os.chdir('/content/rvc')

# 前処理
!python trainset_preprocess_pipeline_print.py \
    dataset/{MODEL_NAME} 40000 2 \
    logs/{MODEL_NAME} True

print('✓ 前処理完了')

In [ ]:
# ピッチ抽出 (rmvpe 推奨)
!python extract_f0_print.py \
    logs/{MODEL_NAME} 2 rmvpe

print('✓ ピッチ抽出完了')

In [ ]:
# 特徴量抽出
!python extract_feature_print.py \
    cuda:0 1 0 0 logs/{MODEL_NAME} v2

print('✓ 特徴量抽出完了')

## Step 4: モデル学習

In [ ]:
# 学習設定
TOTAL_EPOCH = 200      # エポック数 (データ量に応じて調整: 100-300)
SAVE_EVERY_EPOCH = 50  # 保存間隔
BATCH_SIZE = 7         # バッチサイズ (GPU VRAM に応じて調整)

# filelist 生成
!python prepare_list_for_train.py \
    --exp_dir logs/{MODEL_NAME} \
    --sr 40000 \
    --version v2

# 学習実行
!python train_nsf_sim_cache_sid_load_pretrain.py \
    -e {MODEL_NAME} \
    -sr 40k \
    -f0 1 \
    -bs {BATCH_SIZE} \
    -g 0 \
    -te {TOTAL_EPOCH} \
    -se {SAVE_EVERY_EPOCH} \
    -pg pretrained_v2/f0G40k.pth \
    -pd pretrained_v2/f0D40k.pth \
    -l 1 \
    -c 0 \
    -sw 0 \
    -v v2

print('✓ 学習完了')

## Step 5: インデックスの作成

In [ ]:
!python train_index.py \
    --exp_dir logs/{MODEL_NAME} \
    --version v2

print('✓ インデックス作成完了')

## Step 6: モデルを Drive に保存

In [ ]:
import shutil
from pathlib import Path
import glob

save_dir = Path(f"/content/drive/MyDrive/rvc_models/{MODEL_NAME}")
save_dir.mkdir(parents=True, exist_ok=True)

# 最新の .pth モデルをコピー
log_dir = Path(f"/content/rvc/logs/{MODEL_NAME}")
pth_files = sorted(log_dir.glob("*.pth"))
if pth_files:
    latest = pth_files[-1]
    shutil.copy(latest, save_dir / f"{MODEL_NAME}.pth")
    print(f"✓ モデル保存: {latest.name}")

# インデックスファイルをコピー
index_files = list(log_dir.glob("*.index"))
for idx_file in index_files:
    shutil.copy(idx_file, save_dir / idx_file.name)
    print(f"✓ インデックス保存: {idx_file.name}")

print(f"\n保存先: {save_dir}")
print("\n次のステップ:")
print("1. Drive から .pth と .index ファイルをダウンロード")
print("2. ローカルの voice_clone/models/ に配置")
print("3. python3 4_generate_speech.py --text '読み上げるテキスト' を実行")

## (オプション) 動作確認 - 変換テスト

In [ ]:
# edge-tts でテスト音声を生成してRVC変換
!pip install edge-tts -q

import asyncio
import edge_tts

TEST_TEXT = "こんにちは。声質クローンのテストです。"

async def tts():
    communicate = edge_tts.Communicate(TEST_TEXT, "ja-JP-NanamiNeural")
    await communicate.save("/content/test_tts.mp3")
    print("✓ TTS生成完了")

await tts()

In [ ]:
# RVC推論でTTS音声に声質変換を適用
import sys
sys.path.append('/content/rvc')

from infer.modules.vc.modules import VC
import torch

# 最新モデルを使用
log_dir = Path(f"/content/rvc/logs/{MODEL_NAME}")
pth_files = sorted(log_dir.glob("G_*.pth"))
index_files = list(log_dir.glob("*.index"))

if pth_files and index_files:
    vc = VC()
    vc.get_vc(str(pth_files[-1]))
    
    _, wav_opt = vc.vc_single(
        0,
        "/content/test_tts.mp3",
        0,           # f0_up_key (ピッチ変換)
        None,
        "rmvpe",     # f0_method
        str(index_files[0]),
        "",
        0.75,        # index_rate
        3,           # filter_radius
        0,           # resample_sr
        0.25,        # rms_mix_rate
        0.33,        # protect
    )
    
    import soundfile as sf
    sf.write("/content/test_rvc_output.wav", wav_opt[1], wav_opt[0])
    print("✓ RVC変換完了: /content/test_rvc_output.wav")
    
    # 再生
    from IPython.display import Audio, display
    print("元音声 (TTS):")
    display(Audio("/content/test_tts.mp3"))
    print("変換後 (クローン声質):")
    display(Audio("/content/test_rvc_output.wav"))
else:
    print("⚠ モデルが見つかりません。学習を先に完了してください")